# 10 · Front-end impairments

Between the DSP and the channel sit analog components: the IQ modulator,
DACs, drivers, the 90° hybrid, photodiodes and ADCs.  Their imperfections
are deterministic and mostly simple.  This notebook covers the classic
ones, each with how a receiver removes it:

1. **IQ imbalance**: unequal gains and a non-90° angle between I and Q,
   and a time skew between them;
2. **DC offsets**;
3. **limited electrical bandwidth**, which causes ISI;
4. **carrier frequency offset** between the transmitter laser and the LO.
5. **the ADC**: quantization and sampling jitter.

The chain where they all appear together is
[23 · End-to-end coherent transceiver](23-coherent-transceiver-end-to-end.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device
from commkit.impairments import GramSchmidt, Lowdin
from commkit.recovery import ConstantPhase

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)


def host(a):
    return to_device(a, "cpu")


def snr_db(y):
    """SNR after removing a static rotation against the known symbols.

    Metrics never rotate on their own; ConstantPhase() fits one common phase.
    """
    return ck.metrics.snr(ck.recovery.correct_carrier_phase(y, ConstantPhase()))

## 1. IQ imbalance

An ideal receiver measures `I = Re{r}` and `Q = Im{r}` with equal gains on
orthogonal axes.  A real one has an amplitude imbalance `ε` (in dB) and a
phase error `φ`, which mixes in the **mirror image** of the signal:

$$r = \mu\, s + \nu\, s^*,$$

with an image-rejection ratio `|μ|²/|ν|²` of about 25-30 dB for typical
devices (the example below exaggerates it).  The square grid of a QAM constellation becomes a skewed
parallelogram.

In [ ]:
tx = ck.generate(C16, num_symbols=2**15, symbol_rate=32e9, rng=1, device=DEVICE)
rx = ck.impairments.apply_awgn(tx, esn0_db=25, rng=2)
iq = ck.impairments.apply_iq_imbalance(
    rx, amplitude_imbalance_db=1.5, phase_imbalance_deg=8.0
)

fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(rx, ax=axes[0], title="Balanced receiver")
ck.plotting.plot_constellation(iq, ax=axes[1], title="1.5 dB, 8° IQ imbalance")
plt.show()

# The image term: a single tone at +f acquires a mirror at -f.
fs = 1e6
t = np.arange(2**14) / fs
tone = ck.backend.to_device(np.exp(2j * np.pi * 100e3 * t).astype(np.complex64), DEVICE)
tone_iq = ck.impairments.apply_iq_imbalance(
    tone, amplitude_imbalance_db=1.5, phase_imbalance_deg=8.0
)
spec = np.abs(np.fft.fft(host(tone_iq))) ** 2
k = int(round(100e3 / fs * t.size))
print(f"image rejection ratio: {10 * np.log10(spec[k] / spec[-k]):.1f} dB")
_, ax = ck.plotting.plot_psd(
    tone, sampling_rate=fs, nperseg=1024, title=None, label="balanced"
)
ck.plotting.plot_psd(
    tone_iq,
    sampling_rate=fs,
    nperseg=1024,
    ax=ax,
    title="A 100 kHz tone and its mirror image at -100 kHz",
    label="1.5 dB, 8° imbalance",
    legend=True,
    show=True,
)

### Blind correction

Because a QAM (or any circular) signal has I and Q of equal power and
uncorrelated, the imbalance can be removed blindly by re-orthogonalizing
the two branches:

* `GramSchmidt()` keeps I as the reference, removes its projection from Q
  and normalizes both (the classic GSOP);
* `Lowdin()` treats both branches symmetrically (whitening of the 2x2
  covariance), spreading the correction evenly.

Both restore the SNR.  They differ by a small common rotation (GSOP keeps
the I axis, Löwdin splits the difference), which carrier recovery removes
anyway; the SNRs below are measured after a data-aided derotation.

The constellations below show the impaired symbols and the Gram-Schmidt
output: the parallelogram becomes a square grid again, slightly rotated
(the rotation is the carrier's business, not the imbalance's).

In [ ]:
print(f"SNR balanced         {snr_db(rx):5.2f} dB")
print(f"SNR with imbalance   {snr_db(iq):5.2f} dB")
for how in (GramSchmidt(), Lowdin()):
    fixed = ck.impairments.correct_iq_imbalance(iq, how)
    print(f"SNR after {type(how).__name__:12s} {snr_db(fixed):5.2f} dB")

fixed = ck.impairments.correct_iq_imbalance(iq, GramSchmidt())
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(iq, ax=axes[0], title="Before IQ correction")
ck.plotting.plot_constellation(fixed, ax=axes[1], title="After Gram-Schmidt")
plt.show()

### Time skew

The I and Q paths of a receiver (the hybrid's outputs, cables, the ADC
channels) never have exactly equal length.  A **skew** `τ` delays Q against
I.  Unlike the gain and phase errors it depends on frequency: a tone at `f`
sees the phase error `2π f τ`, and its image has the relative power
`tan²(π f τ)`, so the band edges suffer most.  `apply_iq_imbalance` takes it
as `skew=` in seconds, with the sampling rate from the Signal.

Below, 4 ps (an eighth of a 32 GBd symbol) on tones across the band, then on
16-QAM at 2 samples per symbol, where it costs 9 dB: besides the image, Q
is sampled an eighth of a symbol away from the centre of its eye.  The
blind corrections are memoryless: they
see the covariance of I and Q at one instant and cannot undo a delay, so
Gram-Schmidt leaves the SNR where the skew put it.  Removing skew takes a
filter per branch, a deskew filter or a widely linear equalizer.

In [ ]:
SKEW = 4e-12  # 4 ps
fs = 64e9
n = 2**12
t = np.arange(n) / fs
print("tone [GHz]  image rejection [dB]  1/tan²(π f τ) [dB]")
for f_ghz in (2, 8, 16):
    f = round(f_ghz * 1e9 / fs * n) * fs / n  # on an FFT bin
    tone = to_device(np.exp(2j * np.pi * f * t).astype(np.complex64), DEVICE)
    r = ck.impairments.apply_iq_imbalance(
        tone,
        amplitude_imbalance_db=0.0,
        phase_imbalance_deg=0.0,
        skew=SKEW,
        sampling_rate=fs,
    )
    spec = np.abs(np.fft.fft(host(r))) ** 2
    k = round(f / fs * n)
    print(
        f"{f / 1e9:8.0f}    {10 * np.log10(spec[k] / spec[-k]):18.1f}"
        f"  {-20 * np.log10(np.tan(np.pi * f * SKEW)):18.1f}"
    )

qam = ck.impairments.apply_awgn(
    ck.generate(
        C16,
        num_symbols=2**15,
        symbol_rate=32e9,
        sps=2,
        pulse=RRC(rolloff=0.1),
        rng=3,
        device=DEVICE,
    ),
    esn0_db=25,
    rng=4,
)
skewed = ck.impairments.apply_iq_imbalance(
    qam, amplitude_imbalance_db=0.0, phase_imbalance_deg=0.0, skew=SKEW
)


def receive(x):
    return ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))


print(f"\nSNR without skew      {snr_db(receive(qam)):5.2f} dB")
print(f"SNR with 4 ps skew    {snr_db(receive(skewed)):5.2f} dB")
deskewed = ck.impairments.correct_iq_imbalance(skewed, GramSchmidt())
print(f"after Gram-Schmidt    {snr_db(receive(deskewed)):5.2f} dB")

## 2. DC offsets

Imperfectly balanced photodiode pairs and ADC offsets add a constant to I
and Q: a spectral line at 0 Hz and a shifted constellation.  It is removed
by subtracting the mean, *before* the IQ-imbalance correction (which assumes
a zero-mean signal).  In self-homodyne measurements the line sits in the
middle of the signal and needs a dark capture instead (see
[27](27-laser-linewidth-self-homodyne-iq.ipynb)).

In [ ]:
dc = 0.15 - 0.1j
off = iq.replace(samples=iq.samples + dc)
restored = ck.impairments.correct_iq_imbalance(
    off.replace(samples=off.samples - off.samples.mean()), GramSchmidt()
)
print(f"estimated DC {complex(off.samples.mean()):.3f} (true {dc})")
print(
    f"SNR: with DC and imbalance {snr_db(off):.2f} dB, corrected {snr_db(restored):.2f} dB"
)
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(off, ax=axes[0], title="DC offset and IQ imbalance")
ck.plotting.plot_constellation(restored, ax=axes[1], title="Mean removed, then GSOP")
plt.show()

## 3. Limited electrical bandwidth

DACs, drivers and photodiodes low-pass filter the waveform.  A bandwidth
approaching half the symbol rate widens each pulse into its neighbours,
and the eye closes: ISI.  Modeled here as a 4th-order Bessel response
(03 · section 3) with its -3 dB point at `cutoff` (`norm="mag"`), on a
32 GBd RRC signal at 2 samples per symbol.  The adaptive
equalizer removes this ISI later
([17 · Adaptive equalization](17-adaptive-equalization.ipynb)).

In [ ]:
tx2 = ck.generate(
    C16,
    num_symbols=2**15,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=3,
    device=DEVICE,
)


def bandwidth_limit(sig, cutoff):
    """A causal 4th-order Bessel front end.  Its bulk (DC) group delay,
    which timing recovery removes in a receiver, is taken out here."""
    sos = ck.filtering.bessel_sos(
        sampling_rate=sig.sampling_rate, cutoff=cutoff, order=4, norm="mag"
    )
    b, a = sos[:, :3], sos[:, 3:]
    k = np.arange(3)
    delay = np.sum(b @ k / b.sum(axis=1) - a @ k / a.sum(axis=1))  # samples
    filtered = ck.filtering.iir_filter(sig, sos)
    return ck.timing.fft_fractional_delay(filtered, delay=-delay)


fig, axes = plt.subplots(
    1, 3, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 0.75))
)
_, ax_psd = plt.subplots()
for ax, bw in zip(axes, (None, 20e9, 12e9), strict=True):
    x = tx2 if bw is None else bandwidth_limit(tx2, bw)
    y = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))
    label = "no bandwidth limit" if bw is None else f"Bessel {bw / 1e9:.0f} GHz"
    print(f"{label:20s} signal-to-ISI {ck.metrics.snr(y):5.1f} dB")
    ck.plotting.plot_constellation(y, ax=ax, title=label)
    ck.plotting.plot_psd(
        x,
        nperseg=1024,
        ax=ax_psd,
        title="The front end attenuates the band edges",
        label=label,
        legend=True,
    )
plt.show()

## 4. Carrier frequency offset

The transmitter laser and the LO are independent, and their frequencies
differ by up to a few GHz.  The received baseband is shifted by that
offset `Δf`: the constellation **spins** at `Δf` turns per second, which
over one symbol is `2π Δf T` radians.  Even 100 MHz at 32 GBd is a full
turn every 320 symbols, far faster than phase noise; frequency-offset
estimation removes it first
([13 · Frequency-offset estimation](13-frequency-offset-estimation.ipynb)).

In [ ]:
spun = ck.spectral.shift_frequency(tx, frequency=100e6)
phase = np.unwrap(np.angle(host(spun.samples) * np.conj(host(tx.samples))))
slope = np.polyfit(np.arange(phase.size), phase, 1)[0]
print(
    f"phase advance per symbol {slope:.5f} rad, 2π Δf T = {2 * np.pi * 100e6 / 32e9:.5f} rad"
)

fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(
    spun.samples[:100], ax=axes[0], title="First 100 symbols", constellation=C16
)
ck.plotting.plot_constellation(spun, ax=axes[1], title="Whole record: a ring")
plt.show()

## 5. The ADC: quantization and sampling jitter

The ADC rounds each component to one of `2^b` levels between `±full_scale`
(`apply_quantization`).  Its rounding error has the variance `Δ²/12`,
`Δ = 2·full_scale/2^b`, which gives a full-scale sine the classic
`6.02·b + 1.76` dB.  A modulated signal is not a sine: its peaks are rare,
so the full scale is set a few times its rms (the **loading**).  Too low a
full scale clips the peaks, too high a one wastes levels on them.  Below,
the SNR of 16-QAM at 2 samples per symbol for 3 to 8 bits and full scales of
2, 3 and 4 times the rms of a component; without quantization the record
gives 44.5 dB (the pulse's truncation).  The best loading rises with the
resolution: twice the rms for 3 to 5 bits, where the rounding dominates,
three times for 6 and 8 bits, where clipping does.

The sampling instants of a real ADC also jitter around the clock's grid
(`apply_sampling_jitter`, rms `σ` in seconds).  A tone at `f` then carries
noise of relative power `(2π f σ)²`, so the **effective number of bits**,
`ENOB = (SNR - 1.76)/6.02`, falls with the input frequency.  The datasheet's
`enob=` adds the converter's other noise, lumped, on top of the rounding.

In [ ]:
rms = float(np.sqrt(np.mean(host(tx2.samples).real ** 2)))  # one component


def sliced_snr(x):
    return ck.metrics.snr(
        ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))
    )


print("bits   2 x rms   3 x rms   4 x rms   [SNR in dB]")
for bits in (3, 4, 5, 6, 8):
    row = [
        sliced_snr(
            ck.impairments.apply_quantization(tx2, num_bits=bits, full_scale=load * rms)
        )
        for load in (2.0, 3.0, 4.0)
    ]
    print(f"{bits:4d}  " + "  ".join(f"{s:8.2f}" for s in row))

fs = 64e9
t = np.arange(2**14) / fs
print("\ntone [GHz]  SNR [dB]  ENOB   (8 bits, 100 fs rms jitter)")
for f in (1.03e9, 4.07e9, 16.1e9, 30.3e9):
    tone = to_device((0.99 * np.exp(2j * np.pi * f * t)).astype(np.complex64), DEVICE)
    jittered = ck.impairments.apply_sampling_jitter(
        tone, jitter=100e-15, sampling_rate=fs, rng=1
    )
    adc = ck.impairments.apply_quantization(jittered, num_bits=8, full_scale=1.0)
    snr = 10 * np.log10(
        np.mean(np.abs(host(tone)) ** 2) / np.mean(np.abs(host(adc - tone)) ** 2)
    )
    print(f"{f / 1e9:9.1f}  {snr:8.2f}  {(snr - 1.76) / 6.02:5.2f}")

**Summary**

| impairment | signature | removed by |
|---|---|---|
| IQ imbalance | image at `-f`, skewed grid | `correct_iq_imbalance(GramSchmidt() or Lowdin())` |
| IQ skew | image growing with `f` | a deskew filter or widely linear equalizer |
| DC offset | line at 0 Hz, shifted grid | mean subtraction (or a dark capture) |
| bandwidth limit | ISI, closed eye | adaptive equalizer |
| frequency offset | spinning constellation | `correct_frequency_offset` |
| quantization | noise of `Δ²/12`, clipped peaks | resolution and loading |
| sampling jitter | noise growing with `f` | a cleaner clock |

**Next:** [11 · Frame synchronization and timing](11-frame-synchronization-and-timing.ipynb).